# Round 1: ellipsoid on AoU's premade PCs around a 1000G superpopulation

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run, repo_root
from aoucov.batch import run_script
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = cfg.params.get("threads", 4)

In [ ]:
from ancestry import parse_pcs, read_1kg_labels, mahalanobis, fit_quantile_gate, coverage_curve, gate_ellipse
from stage_helpers import hexbin_safe
STAGE = "ancestry_round1"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
REF_POPS, N_PCS, WIDTH = cfg.param("r1_ref_pops"), cfg.param("r1_n_pcs"), cfg.param("r1_width")

## Load

In [ ]:
aou = pd.read_csv(cfg.input("ancestry_preds"), sep="\t", usecols=["research_id", "pca_features"],
                  dtype={"research_id": str})
X = parse_pcs(aou.pop("pca_features"))
ref = pd.read_csv(cfg.input("training_pca"), sep="\t", usecols=["s", "scores"], dtype={"s": str})
R = parse_pcs(ref.pop("scores"))
ref = ref.merge(read_1kg_labels(cfg.input("ref_labels")), on="s", how="left", validate="one_to_one")
rows = (ref["pop"].isin(REF_POPS) & ref["founder"].eq(True)).to_numpy()

missing = sorted(set(REF_POPS) - set(ref["pop"].dropna()))
rep.check("reference populations all present", not missing, ", ".join(missing))
rep.check("gate fit on founders only", True,
          f"{int((ref['pop'].isin(REF_POPS) & ref['founder'].eq(False)).sum())} non-founders left out")
rep.check("enough founders for the covariance", rows.sum() > 10 * N_PCS, f"{rows.sum()} founders, {N_PCS} PCs")

## Gate

In [ ]:
g = fit_quantile_gate(R[rows, :N_PCS], WIDTH)
d = mahalanobis(X, g["mu"], g["C"])
keep = d <= g["radius"]
rep.note(ref_pops=",".join(REF_POPS), n_pcs=N_PCS, width=WIDTH, radius=round(g["radius"], 3))
rep.count("n_participants", len(X))
print("kept", rep.count("n_kept", keep.sum()))

In [ ]:
cc = coverage_curve(d, g["ref_d"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(cc["coverage"] * 100, cc["kept"].where(cc["kept"] > 20))
ax.axvline(WIDTH * 100, color="0.6", ls=":")
ax.set(yscale="log", xlabel="reference founders inside (%)", ylabel="participants inside")
rep.fig(fig, "kept_vs_coverage", "plateau = cluster, steady growth = continuum"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (i, j) in zip(axes, [(0, 1), (2, 3)]):
    hexbin_safe(ax, X[:, i], X[:, j])
    for p in REF_POPS:
        m = rows & ref["pop"].eq(p).to_numpy()
        ax.scatter(R[m, i], R[m, j], s=8, marker="x", label=p)
    gate_ellipse(ax, g["mu"], g["C"], i, j, g["radius"], edgecolor="tab:red", lw=1.5)
    ax.set(xlabel=f"PC{i+1}", ylabel=f"PC{j+1}")
axes[0].legend(fontsize=8)
rep.fig(fig, "gate_pcs"); plt.show()

In [ ]:
founders = ref[ref["founder"].eq(True) & ref["pop"].notna()]
inside = (founders.assign(inside=mahalanobis(R[founders.index], g["mu"], g["C"]) <= g["radius"])
          .groupby(["region", "pop"])["inside"].mean().mul(100).round(1).reset_index(name="pct_inside"))
rep.table(inside, "reference_inside_gate")

In [ ]:
rel = pd.read_csv(cfg.input("relatedness"), sep="\t", dtype={"i.s": str, "j.s": str})
ids = set(aou.loc[keep, "research_id"])
both = rel["i.s"].isin(ids) & rel["j.s"].isin(ids)
cls = pd.cut(rel.loc[both, "kin"], [0.1, 0.177, 0.354, 1.01], right=False,
             labels=["2nd degree", "1st degree", "dup/MZ"])
kin = cls.value_counts().sort_index().rename_axis("class").reset_index(name="pairs")
rep.table(kin, "related_pairs_inside", counts=["pairs"])

## Write

In [ ]:
if needs_run(OUT, FORCE):
    with record(cfg, STAGE, OUT) as meta:
        get_store(OUT).write_text(OUT, "\n".join(aou.loc[keep, "research_id"]) + "\n")
        meta.update(n_kept=int(keep.sum()), radius=g["radius"])
rep.save()